In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

## LER OS RETORNOS, SCORE e SELIC

In [3]:
# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    nome = f'df_ativos_{ano_usado}.csv'
    # caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv('../dados_retornos/otimizacao/ativos/'+nome)
    dfs[ano_usado]['date'] = pd.to_datetime(dfs[ano_usado]['date'])
    dfs[ano_usado] = dfs[ano_usado].set_index('date')
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0).all()])
    print(ano,'-',len(dfs[ano_usado].columns))
    

2025-12-31 - 78
2024-12-31 - 78
2023-12-31 - 78
2022-12-31 - 77
2021-12-31 - 77
2020-12-31 - 73
2019-12-31 - 73
2018-12-31 - 70
2017-12-31 - 70
2016-12-31 - 67
2015-12-31 - 62


In [4]:
dfs['2020']

,ABEV3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,BEEF3,...,SUZB3,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2020-10-01,0.012753,0.083557,-0.001977,0.020059,-0.008726,0.005743,0.001662,0.004642,0.014008,-0.016185,...,0.020856,0.005728,0.045805,0.015130,0.012977,0.003996,-0.004230,-0.002496,0.018264,0.021526
2020-10-02,-0.013380,-0.008496,-0.019497,-0.009021,-0.023286,-0.002691,0.004846,0.008160,-0.008535,-0.025970,...,-0.003655,-0.004272,-0.014487,-0.016721,-0.019980,-0.004962,-0.009175,-0.006593,0.002242,-0.031787
2020-10-05,0.022336,0.010351,0.054599,0.024760,0.046753,0.017506,0.022072,0.014256,0.002460,0.008891,...,0.023095,0.002503,-0.009230,-0.001848,0.012021,0.054883,0.021777,-0.004802,0.046681,0.022502
2020-10-06,0.041377,-0.027894,-0.000958,-0.016344,-0.035698,-0.006619,-0.005943,-0.003511,-0.006133,-0.035251,...,0.011392,-0.001783,0.003103,-0.011114,-0.008267,-0.023650,-0.016277,0.014482,0.018525,-0.010462
2020-10-07,0.014247,-0.004722,0.001918,0.003793,-0.000928,-0.007325,-0.013035,-0.010581,-0.012754,0.007313,...,0.009386,0.003572,-0.010661,0.009367,-0.030207,0.026165,0.026442,-0.013142,0.023642,-0.006561
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-03-25,0.012837,0.053117,0.049551,0.010856,0.018351,0.014405,0.020348,0.022276,0.004639,-0.015083,...,-0.006244,0.001402,0.039348,0.013292,0.035973,-0.003654,-0.007622,0.004443,0.007798,0.038187
2021-03-26,0.010673,-0.020398,-0.003540,0.009595,0.004274,0.003392,0.008238,0.006756,0.004610,0.003063,...,-0.008470,0.008943,-0.025610,-0.001064,0.037697,0.013443,0.033428,0.003099,0.023498,-0.023077
2021-03-29,0.009903,-0.016425,-0.018654,-0.019715,-0.005486,0.004714,0.007308,0.004111,0.004589,0.065114,...,0.006338,0.063711,-0.026667,-0.011001,0.004304,0.005431,0.025647,0.003966,-0.008338,-0.014029


In [5]:
## SELIC para Sigma e excesso
selic_d = pd.read_csv('../dados_retornos/otimizacao/selic/selic_diario.csv').set_index('date')

## EXCESSO DOS ANOS e SIGMA (MAtriz de covariancia)

##### EXCESSO PARA TODOS

In [6]:
dict_excesso = {}
for an in anos:
    ano = an.split("-")[0]
    try:
        print(f"=============== \n EXCESSO {ano}\n ============")
        print("Atualização, Ano: ",ano)
        df = dfs[ano]
        # df_f = pd.DataFrame(eval(df))
        df_f = df.copy()
        slc = selic_d[selic_d.index.isin(df_f.index)]
        slc['valor_diario'] = slc['valor_diario']/100

        print(f"Tamanho DF de {ano}: ", len(df_f))
        print(f"Tamanho Selic: ", len(slc['valor_diario']))
        ano = int(ano)
        dict_excesso[ano] = df_f.sub(slc['valor_diario'],axis=0)
        print("tamanho final do Excesso: ",len(dict_excesso[ano]))
    except Exception as e:
        print(e)
        print("ERror")

 EXCESSO 2025
Atualização, Ano:  2025
Tamanho DF de 2025:  123
Tamanho Selic:  0
tamanho final do Excesso:  123
 EXCESSO 2024
Atualização, Ano:  2024
Tamanho DF de 2024:  122
Tamanho Selic:  0
tamanho final do Excesso:  122
 EXCESSO 2023
Atualização, Ano:  2023
Tamanho DF de 2023:  121
Tamanho Selic:  0
tamanho final do Excesso:  121
 EXCESSO 2022
Atualização, Ano:  2022
Tamanho DF de 2022:  124
Tamanho Selic:  0
tamanho final do Excesso:  124
 EXCESSO 2021
Atualização, Ano:  2021
Tamanho DF de 2021:  123
Tamanho Selic:  0
tamanho final do Excesso:  123
 EXCESSO 2020
Atualização, Ano:  2020
Tamanho DF de 2020:  120
Tamanho Selic:  0
tamanho final do Excesso:  120
 EXCESSO 2019
Atualização, Ano:  2019
Tamanho DF de 2019:  123
Tamanho Selic:  0
tamanho final do Excesso:  123
 EXCESSO 2018
Atualização, Ano:  2018
Tamanho DF de 2018:  119
Tamanho Selic:  0
tamanho final do Excesso:  119
 EXCESSO 2017
Atualização, Ano:  2017
Tamanho DF de 2017:  119
Tamanho Selic:  0
tamanho final do Excess

## Hiperparâmetros

In [7]:
vb_cardinalidade_max = 10
vb_cardinalidade_min = 10
vb_peso_maximo = 0.20
vb_peso_minimo = 0.02
vb_theta = 0.5

### Fazendo otimização ano a ano e comparando com o proximo ano

In [8]:
anos = ['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']
print(anos)

['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']


In [9]:
y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])

print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)

for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    
    # Retornos só para filtrar colunas
    df_usado = dfs[str(ano)]
    retorno_usado = df_usado.copy()    
        #SCORES MF e PIO------------

    score_mf_todos_anos = pd.read_excel(f'../score-raspagem/magic_formula/resultado/{int(ano)}.xlsx')
    score_mf = score_mf_todos_anos[['Unnamed: 0', 'magic_formula']].set_index('Unnamed: 0').T
    
    score_pio_todos_anos = pd.read_excel(f'../score-raspagem/piotroski/resultado/{int(ano)}.xlsx')
    score_pio = score_pio_todos_anos[['Unnamed: 0', 'p_norm']].set_index('Unnamed: 0').T
    
    ativos = retorno_usado.columns.intersection(score_pio.columns).intersection(score_mf.columns)

    score_usado_mf = score_mf[ativos].copy()
    score_usado_pio = score_pio[ativos].copy()

    # print(score_usado)
    print("Score Atualizado")

    ano_um = int(ano)
    # df_usado = f'df_ativos_{ano_um}'
    # retorno_usado = pd.DataFrame(eval(df_usado))
    # df_usado = dfs[str(ano_um)][ativos_finais]
    # retorno_usado = df_usado.copy()
    # retorno_usado = retorno_usado[lista_ativos_finais]
    # print("Retornos atualizados")

    # sigma_usado = retorno_usado.cov()
    # print("Sigmas Atualizados")
    print("-----")
    

    # if df_usado == 'df_ativos_2015':
    #     continue
    # else:
    print(f"{an} TAMANHOS: Retorno {len(ativos)}, Score PIO {len(score_usado_pio.columns)}, Score MF {len(score_usado_mf.columns)}")
    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",int(ano)+1)

    # # print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    # print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    # # print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    # print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = ativos)
    model.ativos = pyo.RangeSet(0, len(ativos)-1)
    # model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    # model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    # model.theta = pyo.Param(initialize=vb_theta)
    model.score_p = pyo.Param(model.ativos, initialize=lambda model,a: score_usado_pio.iloc[0,a])
    model.score_m = pyo.Param(model.ativos, initialize=lambda model,a: score_usado_mf.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos, bounds=(0,1))
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    # model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    # model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    # model.s = pyo.Param(initialize = 2, mutable=True)
    # model.r = pyo.Var(within=pyo.NonNegativeReals)
    
    for i in range(3):
        setattr(model,f'obj{i}_mais',pyo.Var(domain=NonNegativeReals)) #bounds=(0,0.1),
        setattr(model,f'obj{i}_menos',pyo.Var(domain=NonNegativeReals))
    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================

    def func_objetivo_1(model):
       
        return model.obj1_menos + model.obj2_menos
    model.objetivo = pyo.Objective(rule=func_objetivo_1, sense=pyo.minimize)

    #=============================
    # RESTRIÇÕES
    #=============================

    # # OBJ 1 2 3 - Rest 1 2 3
    # def retorno_restr_rule(model,diad):
    #     return sum(model.x[a]*model.retornos_ativos[dia,a] for a in model.ativos for dia in model.dias) + model.obj0_menos - model.obj0_mais >= model.retornos_ibov[diad]
    # model.retorno_restr = pyo.Constraint(model.dias,rule=retorno_restr_rule)

    def piotroski_restr_rule(model):
        return sum(model.x[a]*model.score_p[a] for a in model.ativos) + model.obj1_menos - model.obj1_mais >= 0.9
    model.piotroski_rest = pyo.Constraint(rule=piotroski_restr_rule)

    def mf_restr_rule(model):
        return sum(model.x[a]*model.score_m[a] for a in model.ativos) + model.obj2_menos - model.obj2_mais >= 0.9
    model.mf_restr = pyo.Constraint(rule=mf_restr_rule)

    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    #peso minimo por acao
    def rule_peso_minimo(model, a):
        return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)


    def cardinalidade_min(model):
        return sum(
            model.y[a] for a in model.ativos
            ) >= model.cardinalidade_valor_min
    model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)


    melhor_pesos = {list(model.nome_ativos.data())[a]: pyo.value(model.x[a]) for a in model.ativos}
    pio = {a: sum(pyo.value(model.x[a])*pyo.value(model.score_p[a]) for a in model.ativos)}
    mf = {a: sum(pyo.value(model.x[a])*pyo.value(model.score_m[a]) for a in model.ativos)}
    carteiras_anuais[int(ano_um)+1] = {
        'pesos':  melhor_pesos,
        # 'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_um} -> {melhor_pesos}")
    print(f"{ano_um} Piotroski -> {pio}")
    # print(f"Valores obj Menos: {pyo.value(model.obj0_menos), pyo.value(model.obj1_menos), pyo.value(model.obj2_menos)}")
    print(f"Valores obj Menos: {pyo.value(model.obj1_menos), pyo.value(model.obj1_mais)}")
    print(f"{ano_um} Magic Formula -> {mf}")
    # print(f"Valores obj Menos: {pyo.value(model.obj0_menos), pyo.value(model.obj1_menos), pyo.value(model.obj2_menos)}")
    print(f"Valores obj Menos: {pyo.value(model.obj2_menos), pyo.value(model.obj2_mais)}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass




=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2015
Nao consta model
Score Atualizado
-----
2015-12-31 TAMANHOS: Retorno 50, Score PIO 50, Score MF 50
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
2015 -> {'ABEV3': 0.02, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BEEF3': 0.0, 'CMIG4': 0.0, 'COGN3': 0.02, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.0, 'CSMG3': 0.0, 'CSNA3': 0.0, 'CYRE3': 0.0, 'DIRR3': 0.0, 'ECOR3': 0.0, 'EMBJ3': 0.0, 'ENGI11': 0.0, 'EQTL3': 0.0, 'EZTC3': 0.0, 'FLRY3': 0.20000000000000007, 'GGBR4': 0.0, 'GOAU4': 0.0, 'HYPE3': 0.0, 'ISAE4': 0.02, 'ITSA4': 0.0, 'JHSF3': 0.0, 'KLBN11': 0.0, 'LREN3': 0.2, 'MGLU3': 0.0, 'MOTV3': 0.0, 'MRVE3': 0.02, 'MULT3': 0.0, 'PETR3': 0.0, 'PETR4'

In [10]:
carteiras_anuais

{2016: {'pesos': {'ABEV3': 0.02,
   'ANIM3': 0.0,
   'AXIA3': 0.0,
   'AZZA3': 0.0,
   'B3SA3': 0.0,
   'BEEF3': 0.0,
   'CMIG4': 0.0,
   'COGN3': 0.02,
   'CPFE3': 0.0,
   'CPLE3': 0.0,
   'CSAN3': 0.0,
   'CSMG3': 0.0,
   'CSNA3': 0.0,
   'CYRE3': 0.0,
   'DIRR3': 0.0,
   'ECOR3': 0.0,
   'EMBJ3': 0.0,
   'ENGI11': 0.0,
   'EQTL3': 0.0,
   'EZTC3': 0.0,
   'FLRY3': 0.20000000000000007,
   'GGBR4': 0.0,
   'GOAU4': 0.0,
   'HYPE3': 0.0,
   'ISAE4': 0.02,
   'ITSA4': 0.0,
   'JHSF3': 0.0,
   'KLBN11': 0.0,
   'LREN3': 0.2,
   'MGLU3': 0.0,
   'MOTV3': 0.0,
   'MRVE3': 0.02,
   'MULT3': 0.0,
   'PETR3': 0.0,
   'PETR4': 0.0,
   'POMO4': 0.0,
   'PRIO3': 0.0,
   'RADL3': 0.2,
   'RENT3': 0.0,
   'SBSP3': 0.0,
   'SLCE3': 0.02,
   'SMTO3': 0.0,
   'SUZB3': 0.0,
   'TAEE11': 0.09999999999999992,
   'TOTS3': 0.0,
   'UGPA3': 0.2,
   'USIM5': 0.0,
   'VALE3': 0.0,
   'VIVT3': 0.0,
   'WEGE3': 0.0}},
 2017: {'pesos': {'ABEV3': 0.0,
   'ANIM3': 0.0,
   'AXIA3': 0.0,
   'AZZA3': 0.0,
   'B3SA3'

In [11]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if v >= vb_peso_minimo:
            linhas.append({'ano': an, 'ativo': k, 'peso': round(v, 4)})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [12]:
df_portfolios[df_portfolios['ano'] == 2020]

,ano,ativo,peso
40,2020,BEEF3,0.02
41,2020,CPFE3,0.02
42,2020,CPLE3,0.02
43,2020,CSMG3,0.10
44,2020,LREN3,0.02
45,2020,MOTV3,0.20
46,2020,SAPR11,0.20
47,2020,SBSP3,0.20
48,2020,WEGE3,0.20


In [13]:
df_portfolios.to_csv('../carteiras/goal_programming/carteiras_goal_programming.csv')
